# Sellers - Silver Transformation

## Parameters

In [0]:
dbutils.widgets.text(
    name="environment",
    defaultValue="dev",
    label="Environment"
)

environment = dbutils.widgets.get("environment").strip().lower()

if environment not in ("dev", "test", "prod"):
    raise ValueError(
        f"Unsupported environment: {environment}. Expected dev, test, or prod."
    )

## Setup

In [0]:
from pyspark.sql.functions import col, create_map, lit

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.olist_sellers"
target_table = f"{catalog}.silver.olist_sellers"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Transform to Silver

In [0]:
brazil_state_map = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AM": "Amazonas",
    "AP": "Amapá",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MG": "Minas Gerais",
    "MS": "Mato Grosso do Sul",
    "MT": "Mato Grosso",
    "PA": "Pará",
    "PB": "Paraíba",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "PR": "Paraná",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RO": "Rondônia",
    "RR": "Roraima",
    "RS": "Rio Grande do Sul",
    "SC": "Santa Catarina",
    "SE": "Sergipe",
    "SP": "São Paulo",
    "TO": "Tocantins"
}

In [0]:
state_map_expr = create_map(
    *[
        item
        for state_code, state_name in brazil_state_map.items()
        for item in (lit(state_code), lit(state_name))
    ]
)

silver_df = bronze_df.withColumn(
    "seller_state_name",
    state_map_expr[col("seller_state")]
)

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)